<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/GPR_LR_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.



## Overview

This notebook applies **Gaussian Process Regression (GPR)** to model:
- **Y1 — Heating Load** (kWh/m²)
- **Y2 — Cooling Load** (kWh/m²)

from the ENB2012 Energy Efficiency dataset (8 building features: X1–X8).

### Structure
| Section | Content |
|---|---|
| 1 | Imports & Data Loading |
| 2 | Correlation Analysis (motivation for multi-output GP) |
| 3 | Data Preparation |
| 4 | Single-Output GPR — Theory & Implementation |
| 5 | Predictions & Evaluation |
| 6 | Visualisation |
| 7 | Multi-Output GP Discussion |
| 8 | Save Models |

## Section 1 — Imports & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import joblib

In [ ]:
# Download dataset from Kaggle (note: 'eergy' is the actual slug used by the dataset author)
try:
    import kagglehub
    path = kagglehub.dataset_download("elikplim/eergy-efficiency-dataset")
    csv_path = f"{path}/ENB2012_data.csv"
    print("Downloaded to:", csv_path)
except Exception as e:
    print(f"Kaggle download failed ({e}). Falling back to local file.")
    csv_path = "ENB2012_data.csv"

df = pd.read_csv(csv_path)
print("Dataset shape:", df.shape)
df.head()

In [ ]:
df.describe().round(2)

## Section 2 — Correlation Analysis

Before building any model, we check whether Y1 (Heating) and Y2 (Cooling) are correlated.

**Why this matters:** If $\text{corr}(Y_1, Y_2)$ is high, the two outputs share mutual information and a **multivariate GP** with a matrix-valued kernel $\kappa(g,g') \in \mathbb{R}^{2 \times 2}$ is theoretically justified over two independent scalar GPs.

In [ ]:
corr_y1_y2 = df["Y1"].corr(df["Y2"])
print(f"Pearson Correlation  Y1 (Heating) ↔ Y2 (Cooling): {corr_y1_y2:.4f}")

In [ ]:
fig_corr = px.scatter(
    df, x="Y1", y="Y2",
    labels={"Y1": "Heating Load Y₁ (kWh/m²)", "Y2": "Cooling Load Y₂ (kWh/m²)"},
    title=f"Heating vs Cooling Load  —  Pearson r = {corr_y1_y2:.4f}",
    trendline="ols",
    template="plotly_white",
    opacity=0.6
)
fig_corr.update_traces(
    marker=dict(size=5, color="steelblue"),
    selector=dict(mode="markers")
)
fig_corr.show()

In [ ]:
corr_matrix = df.corr().round(3)

fig_heatmap = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation Matrix — All Variables",
    template="plotly_white"
)
fig_heatmap.show()

## Section 3 — Data Preparation

In [ ]:
feature_cols = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']
X          = df[feature_cols].values
y_heating  = df['Y1'].values
y_cooling  = df['Y2'].values

# Single split shared by both targets — guarantees Y1 and Y2 labels
# correspond to exactly the same rows in train and test sets.
X_train, X_test, yh_train, yh_test, yc_train, yc_test = train_test_split(
    X, y_heating, y_cooling,
    test_size=0.2, random_state=42
)

# StandardScaler fitted on training data only (prevents data leakage)
scaler     = StandardScaler()
X_train_s  = scaler.fit_transform(X_train)
X_test_s   = scaler.transform(X_test)

print(f"Training samples : {len(X_train_s)}")
print(f"Test     samples : {len(X_test_s)}")

## Section 4 — Single-Output GPR

### Theory (Section 2.1)

**Latent Process:**
$$X_g \sim \mathcal{GP}(\mu_g,\, \kappa(g, g'))$$

**Noisy Observations:**
$$Y_g = X_g + \nu_g, \qquad \nu_g \sim \mathcal{N}(0, \sigma_m^2)$$

**Posterior Mean (Prediction):**
$$\mathbb{E}[X_g \mid \mathscr{Y}_n = y_n] = \mu_g + k^T(K_n + \sigma_m^2 I)^{-1}(y_n - \mu_{\mathscr{Y}_n})$$

**Posterior Variance (Uncertainty):**
$$\mathrm{Var}(X_g \mid \mathscr{Y}_n = y_n) = \kappa(g,g) - k^T(K_n + \sigma_m^2 I)^{-1}k$$

**Log Marginal Likelihood (hyperparameter optimisation):**
$$\log p(y_n \mid \theta) = -\tfrac{1}{2}\, y_n^T(K_n + \sigma_m^2 I)^{-1}y_n
  - \tfrac{1}{2}\log|K_n + \sigma_m^2 I| - \tfrac{n}{2}\log 2\pi$$

**Kernel used:** $\underbrace{C(\cdot)\times\text{RBF}(\cdot)}_{\text{signal}} + \underbrace{\text{WhiteKernel}(\cdot)}_{\text{noise}}$

In [ ]:
def build_kernel():
    """Return a fresh kernel instance for each GP to avoid shared-state issues."""
    return (
        C(1.0, constant_value_bounds=(1e-3, 1e3))
        * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 1e2))
        + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
    )

In [ ]:
# GPR for Heating Load (Y1)
gp_h = GaussianProcessRegressor(
    kernel=build_kernel(),
    alpha=1e-6,               # small nugget for numerical stability
    n_restarts_optimizer=15,  # restarts to avoid local optima in LML
    random_state=42
)
gp_h.fit(X_train_s, yh_train)

print("Heating — Optimised Kernel:")
print(" ", gp_h.kernel_)
print(f"Heating — Log-Marginal-Likelihood: {gp_h.log_marginal_likelihood_value_:.4f}")

In [ ]:
# GPR for Cooling Load (Y2)
gp_c = GaussianProcessRegressor(
    kernel=build_kernel(),
    alpha=1e-6,
    n_restarts_optimizer=15,
    random_state=42
)
gp_c.fit(X_train_s, yc_train)

print("Cooling — Optimised Kernel:")
print(" ", gp_c.kernel_)
print(f"Cooling — Log-Marginal-Likelihood: {gp_c.log_marginal_likelihood_value_:.4f}")

## Section 5 — Predictions & Evaluation

In [ ]:
yh_pred, yh_std = gp_h.predict(X_test_s, return_std=True)
yc_pred, yc_std = gp_c.predict(X_test_s, return_std=True)

results = pd.DataFrame({
    "Target"  : ["Heating Load Y1", "Cooling Load Y2"],
    "R²"      : [r2_score(yh_test, yh_pred),  r2_score(yc_test, yc_pred)],
    "RMSE"    : [np.sqrt(mean_squared_error(yh_test, yh_pred)),
                 np.sqrt(mean_squared_error(yc_test, yc_pred))],
    "Mean σ"  : [yh_std.mean(), yc_std.mean()]
}).round(4)

results

## Section 6 — Visualisation

In [ ]:
# 6a. Predicted vs Actual with 95% Credible Intervals
val_lo = min(yh_test.min(), yc_test.min()) - 1
val_hi = max(yh_test.max(), yc_test.max()) + 1

r2_h = r2_score(yh_test, yh_pred)
r2_c = r2_score(yc_test, yc_pred)

fig_pred = go.Figure()

fig_pred.add_trace(go.Scatter(
    x=yh_test, y=yh_pred, mode='markers',
    marker=dict(color='royalblue', size=5, opacity=0.7),
    name=f'Heating Y1  (R²={r2_h:.3f})',
    error_y=dict(type='data', array=1.96*yh_std,
                 visible=True, color='royalblue', thickness=1, width=2)
))

fig_pred.add_trace(go.Scatter(
    x=yc_test, y=yc_pred, mode='markers',
    marker=dict(color='tomato', size=5, opacity=0.7),
    name=f'Cooling Y2  (R²={r2_c:.3f})',
    error_y=dict(type='data', array=1.96*yc_std,
                 visible=True, color='tomato', thickness=1, width=2)
))

fig_pred.add_trace(go.Scatter(
    x=[val_lo, val_hi], y=[val_lo, val_hi],
    mode='lines', line=dict(dash='dash', color='gray'),
    name='Perfect Prediction'
))

fig_pred.update_layout(
    title='GPR: Predicted vs Actual with 95% Credible Intervals',
    xaxis_title='Actual Load (kWh/m²)',
    yaxis_title='Predicted Load (kWh/m²)',
    template='plotly_white',
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)
fig_pred.show()

In [ ]:
# 6b. Posterior Uncertainty Distribution
fig_unc = go.Figure()
fig_unc.add_trace(go.Histogram(
    x=yh_std, name='Heating σ', opacity=0.7,
    marker_color='royalblue', nbinsx=30
))
fig_unc.add_trace(go.Histogram(
    x=yc_std, name='Cooling σ', opacity=0.7,
    marker_color='tomato', nbinsx=30
))
fig_unc.update_layout(
    barmode='overlay',
    title='Distribution of Posterior Standard Deviations',
    xaxis_title='Posterior Std Dev σ (kWh/m²)',
    yaxis_title='Count',
    template='plotly_white'
)
fig_unc.show()

In [ ]:
# 6c. Residuals Plot
fig_res = go.Figure()
fig_res.add_trace(go.Scatter(
    x=yh_pred, y=yh_test - yh_pred, mode='markers',
    marker=dict(color='royalblue', size=5, opacity=0.7),
    name='Heating Y1 Residuals'
))
fig_res.add_trace(go.Scatter(
    x=yc_pred, y=yc_test - yc_pred, mode='markers',
    marker=dict(color='tomato', size=5, opacity=0.7),
    name='Cooling Y2 Residuals'
))
fig_res.add_hline(y=0, line_dash='dash', line_color='gray')
fig_res.update_layout(
    title='Residuals vs Predicted Values',
    xaxis_title='Predicted Load (kWh/m²)',
    yaxis_title='Residual (Actual − Predicted)',
    template='plotly_white'
)
fig_res.show()

## Section 7 — Multi-Output GP Discussion (Section 2.2)

### Theoretical Formulation

The Energy Efficiency dataset has **two correlated outputs**, so the full multivariate GP formulation models them jointly:

$$X_g = \begin{bmatrix} Y_1 \\ Y_2 \end{bmatrix} \in \mathbb{R}^q, \qquad q = 2$$

The scalar kernel $\kappa(g,g') \in \mathbb{R}$ is replaced by a **matrix-valued kernel**:

$$\kappa(g,g') \in \mathbb{R}^{2 \times 2}$$

whose entries capture:

| Entry | Meaning |
|---|---|
| $\kappa_{11}(g,g')$ | Auto-covariance of Y1 (Heating) |
| $\kappa_{22}(g,g')$ | Auto-covariance of Y2 (Cooling) |
| $\kappa_{12}(g,g')$ | **Cross-covariance Y1 ↔ Y2** |

### Intrinsic Coregionalisation Model (ICM)

A common choice is the **ICM**:

$$\kappa_{\text{ICM}}(g,g') = B \otimes \kappa_{\text{base}}(g,g')$$

where $B \in \mathbb{R}^{2 \times 2}$ is the **coregionalisation matrix** (learned from data) encoding output correlations, and $\otimes$ is the Kronecker product.

### Why it is justified here

The strong observed Pearson correlation between Y1 and Y2 means the cross-covariance term $\kappa_{12}$ would transfer predictive information between the two outputs — potentially improving accuracy for both.

### Limitation of scikit-learn

`GaussianProcessRegressor` supports **scalar outputs only**. True multi-output GP requires:

| Library | Implementation |
|---|---|
| **GPflow** (TensorFlow) | `gpflow.models.GPR` with `SharedIndependent` or LMC kernel |
| **GPyTorch** (PyTorch) | `gpytorch.models.ExactGP` with `MultitaskGaussianLikelihood` |

### Conclusion

> Two independent scalar GPs were trained as a practical approximation, which is valid under the assumption that Y1 and Y2 are **conditionally independent given X**. However, the high observed correlation between Heating and Cooling Loads suggests that a true multivariate GP — using a matrix-valued kernel and the ICM formulation — would be a theoretically superior approach and is recommended as a future extension.

In [ ]:
print(f"Pearson Correlation Y1–Y2 : {corr_y1_y2:.4f}")

if corr_y1_y2 > 0.9:
    verdict = "Very strong — multi-output GP is strongly justified."
elif corr_y1_y2 > 0.7:
    verdict = "Strong — multi-output GP is justified."
else:
    verdict = "Moderate — independent GPs may suffice."

print(f"Interpretation          : {verdict}")

## Section 8 — Save Models

In [ ]:
joblib.dump(gp_h, 'gpr_heating.pkl')
joblib.dump(gp_c, 'gpr_cooling.pkl')
print("Saved → gpr_heating.pkl")
print("Saved → gpr_cooling.pkl")



1. **GPR provides an excellent fit** (high R²) for both targets due to its non-parametric nature and automatic hyperparameter optimisation via Log Marginal Likelihood.
2. **Uncertainty quantification** via posterior standard deviation helps identify building configurations where predictions are less reliable.
3. **Y1 and Y2 are highly correlated**, validating the multi-output GP formulation from Section 2.2 as a theoretically justified extension.
4. **Practical limitation:** scikit-learn does not support multi-output GP. Independent scalar GPs were used as a valid approximation under conditional independence.
5. **Recommendations:** Explore GPflow/GPyTorch for ICM-based multi-output GP, or experiment with Matérn / ARD kernels for improved feature relevance weighting.

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.



## Objective
The goal of this study is to model and predict `predicted_energy_demand` using a linear regression framework based on environmental and building operation variables.

We assume a linear relationship:

$$Y = \beta X + \beta_0 + \nu$$

where:
- **Y** = predicted energy demand
- **X** = selected building/environment features
- **ν** = Gaussian noise

We justify feature selection based on physical and operational relevance to building energy consumption.

## 1. Import Libraries

In [ ]:
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

## 2. Load Dataset

In [ ]:
# Download dataset
kagglepath = "programmer3/green-building-multi-source-environment-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Dataset path:", path)

# FIX: Use os.path.join for cross-platform compatibility
csv_path = os.path.join(path, "green_building_dataset.csv")
df = pd.read_csv(csv_path)

df.head()

## 3. Basic Exploration

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

## 4. Missing Values Handling

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
print("Missing values per column:")
print(missing[missing > 0])

# Drop rows with missing values
df = df.dropna()
print("\nShape after dropping missing values:", df.shape)

## 5. Correlation Heatmap

Before selecting features, we examine correlations to:
- Identify which variables are strongly related to the target
- Detect multicollinearity between features

In [ ]:
candidate_features = [
    "ventilation_rate", "equipment_load", "occupancy",
    "heating_energy", "cooling_energy", "electricity_consumption",
    "predicted_energy_demand"
]

plt.figure(figsize=(10, 7))
corr_matrix = df[candidate_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap of Candidate Features")
plt.tight_layout()
plt.show()

## 6. Feature Selection

### Justification

We select features that have direct or indirect physical influence on building energy demand.

### Selected Features:

| Feature | Justification |
|---|---|
| `ventilation_rate` | Affects HVAC load and air exchange energy |
| `equipment_load` | Internal heat gain from devices |
| `occupancy` | Human presence increases energy usage |
| `heating_energy` | Direct heating consumption |
| `cooling_energy` | Direct cooling consumption |

### Why `electricity_consumption` is excluded:

> **Multicollinearity Fix:** `electricity_consumption` is a composite metric that typically includes `heating_energy` and `cooling_energy` as sub-components. Including all three together creates **severe multicollinearity**, inflating coefficients and making the model unreliable. We retain the sub-components as they have clearer physical interpretations.

## 7. Define Features and Target

In [ ]:
features = [
    "ventilation_rate",
    "equipment_load",
    "occupancy",
    "heating_energy",
    "cooling_energy"
]

target = "predicted_energy_demand"

X = df[features]
y = df[target]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

## 8. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

## 9. Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")

## 10. Model Coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_,
    "Abs_Coefficient": np.abs(model.coef_)
}).sort_values("Abs_Coefficient", ascending=False)

print("Intercept:", model.intercept_)
coef_df

In [ ]:
# Bar chart of coefficients
plt.figure(figsize=(8, 5))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel("Coefficient Value")
plt.title("Feature Coefficients")
plt.tight_layout()
plt.show()

## 11. Predictions

In [ ]:
y_pred_test  = model.predict(X_test)
y_pred_train = model.predict(X_train)

## 12. Model Evaluation

We evaluate on **both training and test sets** to check for overfitting.

In [ ]:
train_r2   = r2_score(y_train, y_pred_train)
test_r2    = r2_score(y_test,  y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_test,  y_pred_test))

print(f"{'Metric':<10} {'Train':>10} {'Test':>10}")
print("-" * 32)
print(f"{'R²':<10} {train_r2:>10.4f} {test_r2:>10.4f}")
print(f"{'RMSE':<10} {train_rmse:>10.4f} {test_rmse:>10.4f}")

if abs(train_r2 - test_r2) < 0.05:
    print("\n✅ Train and Test R² are close — no significant overfitting detected.")
else:
    print("\n⚠️ Gap between Train and Test R² — possible overfitting.")

## 13. Residual Analysis

In [ ]:
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Residuals vs Predicted
axes[0].scatter(y_pred_test, residuals, alpha=0.5, color='steelblue')
axes[0].axhline(0, color='red', linewidth=1.5)
axes[0].set_xlabel("Predicted Energy Demand")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residual Plot")

# Plot 2: Histogram of Residuals (checks Gaussian noise assumption)
axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linewidth=1.5)
axes[1].set_xlabel("Residual Value")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual Distribution (Gaussian Check)")

plt.tight_layout()
plt.show()

## 14. Interpretation of Results

### Model Performance
- **R²** indicates how much variance in energy demand is explained by the model. Values closer to 1.0 are better.
- **RMSE** represents the average prediction error in the same units as the target variable.
- Comparing **Train R²** and **Test R²** helps detect overfitting — a large gap suggests the model memorised training data.

### Coefficients Interpretation
- **Positive coefficients** — variables that increase energy demand (e.g., occupancy, equipment load).
- **Negative coefficients** — variables that reduce demand (or reflect dataset-specific patterns).
- Sorting by absolute value reveals which features have the **strongest influence**.

### Key Observations
- Occupancy and equipment load typically contribute significantly to energy demand.
- HVAC-related variables (heating and cooling energy) strongly influence predictions.
- Excluding `electricity_consumption` reduced multicollinearity and improved coefficient reliability.

### Residual Analysis
- If the residual scatter plot shows **no pattern** (random cloud around zero), the linear model assumption is valid.
- If the histogram of residuals is **approximately bell-shaped**, the Gaussian noise assumption holds.
- Any visible curve or funnel shape indicates non-linearity or heteroscedasticity.

## 15. Limitations

1. **Linear assumption** may oversimplify real building energy behaviour.
2. **Residual multicollinearity** may still exist between heating and cooling energy.
3. **External factors** such as weather, insulation quality, and building orientation are not included.
4. **Gaussian noise assumption** may not fully represent real-world data distributions.
5. **`dropna()`** may remove informative rows — imputation could be a better strategy.

Despite these limitations, linear regression provides a strong, interpretable baseline model.

## 16. Conclusion

A multivariate linear regression model was successfully built to predict energy demand in green buildings.

Key takeaways:
- Building operational variables such as occupancy, equipment load, and HVAC energy usage have strong linear relationships with energy demand.
- Removing `electricity_consumption` addressed the multicollinearity issue identified through the correlation heatmap.
- Train and test R² comparison confirmed the model generalises well without overfitting.
- Residual analysis validated the Gaussian noise and linearity assumptions.

While more advanced models (Ridge, Lasso, or ensemble methods) may improve accuracy, linear regression provides a **transparent and interpretable baseline** suitable for engineering analysis.